# FocusAI — Análisis Exploratorio de Datos (EDA)

**Autor:** Luis Vasquez — Model Builder  
**Proyecto:** Pipeline MLOps para clasificar entradas de diario como *Productivo* / *Procrastinación*

---

## Contenido
1. Setup e imports
2. Visión general del dataset
3. Distribución de clases
4. Análisis de longitud de texto
5. Frecuencia de palabras por clase
6. Pipeline NLP: texto crudo vs. texto limpio
7. Términos TF-IDF más relevantes por clase
8. Resultados del modelo (métricas CV y hold-out)
9. Matriz de confusión
10. Distribución de probabilidades (efecto calibración)

---
## 1. Setup e imports

In [ ]:
import json
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Asegurar que el root del proyecto esté en sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    CLEANED_CSV_PATH,
    HOLDOUT_METRICS_PATH,
    LABEL_PROCRASTINATION,
    LABEL_PRODUCTIVE,
    METRICS_PATH,
    PER_CLASS_METRICS_PATH,
    RAW_CSV_PATH,
    VECTORIZER_PATH,
    VECTORIZED_CSV_PATH,
)

# Paleta de colores del proyecto
PALETTE = {
    LABEL_PRODUCTIVE: '#2ecc71',      # verde
    LABEL_PROCRASTINATION: '#e74c3c', # rojo
}
COLORS = [PALETTE[LABEL_PRODUCTIVE], PALETTE[LABEL_PROCRASTINATION]]

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Setup OK — PROJECT_ROOT:', PROJECT_ROOT)

---
## 2. Visión general del dataset

In [ ]:
df_raw = pd.read_csv(RAW_CSV_PATH)

print(f'Filas      : {len(df_raw)}')
print(f'Columnas   : {list(df_raw.columns)}')
print(f'Nulos      : {df_raw.isnull().sum().to_dict()}')
print(f'Duplicados : {df_raw.duplicated().sum()}')
print()
df_raw.head(6)

In [ ]:
# Estadísticas descriptivas del texto
df_raw['n_chars']  = df_raw['texto'].str.len()
df_raw['n_words']  = df_raw['texto'].str.split().str.len()

df_raw.groupby('etiqueta')[['n_chars', 'n_words']].describe().round(1)

---
## 3. Distribución de clases

In [ ]:
counts = df_raw['etiqueta'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Barras
bars = axes[0].bar(counts.index, counts.values,
                   color=[PALETTE.get(l, '#95a5a6') for l in counts.index],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Conteo por clase', fontweight='bold')
axes[0].set_ylabel('Muestras')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Pie
axes[1].pie(
    counts.values,
    labels=counts.index,
    colors=[PALETTE.get(l, '#95a5a6') for l in counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Proporción de clases', fontweight='bold')

plt.suptitle('Balance del dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Balance: {counts.to_dict()}')
print(f'Ratio Productivo/Procrastinación: {counts.iloc[0]/counts.iloc[1]:.2f}')

---
## 4. Análisis de longitud de texto

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in zip(axes, ['n_words', 'n_chars'], ['Palabras por entrada', 'Caracteres por entrada']):
    for clase, color in PALETTE.items():
        subset = df_raw[df_raw['etiqueta'] == clase][col]
        ax.hist(subset, bins=10, alpha=0.6, color=color, label=clase, edgecolor='white')
        ax.axvline(subset.mean(), color=color, linestyle='--', linewidth=1.5)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Distribución de longitud por clase', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot comparativo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in zip(axes, ['n_words', 'n_chars'], ['N° palabras', 'N° caracteres']):
    data_plot = [df_raw[df_raw['etiqueta'] == c][col].values for c in [LABEL_PRODUCTIVE, LABEL_PROCRASTINATION]]
    bp = ax.boxplot(data_plot, patch_artist=True,
                    labels=[LABEL_PRODUCTIVE, LABEL_PROCRASTINATION])
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f'Boxplot — {label}', fontweight='bold')
    ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

---
## 5. Frecuencia de palabras por clase

In [ ]:
import re

STOPWORDS_SIMPLES = {
    'de', 'la', 'el', 'en', 'y', 'a', 'que', 'los', 'las', 'un', 'una',
    'con', 'del', 'por', 'para', 'no', 'se', 'lo', 'al', 'es', 'me',
    'mi', 'su', 'todo', 'pero', 'como', 'más', 'sin', 'fue', 'ni',
    'le', 'si', 'ya', 'he', 'hay', 'era', 'muy', 'has', 'o', 'e',
}

def tokenize_simple(text):
    words = re.findall(r'[a-záéíóúüñ]+', text.lower())
    return [w for w in words if w not in STOPWORDS_SIMPLES and len(w) >= 4]

TOP_N = 15
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, clase, color in zip(axes, [LABEL_PRODUCTIVE, LABEL_PROCRASTINATION], COLORS):
    textos = df_raw[df_raw['etiqueta'] == clase]['texto'].str.cat(sep=' ')
    freq = Counter(tokenize_simple(textos)).most_common(TOP_N)
    words, counts_w = zip(*freq)

    bars = ax.barh(list(reversed(words)), list(reversed(counts_w)),
                   color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'Top {TOP_N} palabras — {clase}', fontweight='bold')
    ax.set_xlabel('Frecuencia')
    for bar, val in zip(bars, list(reversed(counts_w))):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)

plt.suptitle('Palabras más frecuentes por clase (texto crudo)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Pipeline NLP: texto crudo vs. texto limpio

In [ ]:
from src.nlp.preprocess import clean_text, run_nlp_pipeline

# Ejecutar pipeline si no existe el CSV limpio
if not CLEANED_CSV_PATH.exists():
    print('Ejecutando pipeline NLP...')
    run_nlp_pipeline()

df_clean = pd.read_csv(CLEANED_CSV_PATH)

# Mostrar ejemplos antes/después
ejemplos = df_raw[['texto', 'etiqueta']].copy()
ejemplos['texto_limpio'] = df_clean['texto_limpio']
ejemplos['reduccion_%'] = (1 - ejemplos['texto_limpio'].str.len() / ejemplos['texto'].str.len()) * 100

print('Ejemplos de limpieza (texto crudo → texto limpio):\n')
for _, row in ejemplos.head(4).iterrows():
    print(f'[{row["etiqueta"]}]')
    print(f'  CRUDO  : {row["texto"][:80]}...')
    print(f'  LIMPIO : {row["texto_limpio"]}')
    print(f'  Reducción: {row["reduccion_%"]:.0f}%')
    print()

In [ ]:
# Comparativa de longitud antes/después por clase
df_clean['etiqueta_str'] = df_raw['etiqueta']
df_clean['n_words_clean'] = df_clean['texto_limpio'].str.split().str.len()
df_clean['n_words_raw']   = df_raw['n_words']

reduccion = df_clean.groupby('etiqueta_str')[['n_words_raw', 'n_words_clean']].mean().round(1)
reduccion.columns = ['Palabras (crudo)', 'Palabras (limpio)']
reduccion['Reducción %'] = ((1 - reduccion['Palabras (limpio)'] / reduccion['Palabras (crudo)']) * 100).round(1)
print(reduccion)

reduccion[['Palabras (crudo)', 'Palabras (limpio)']].plot(
    kind='bar', figsize=(8, 4),
    color=['#95a5a6', '#3498db'],
    edgecolor='white', title='Promedio de palabras: crudo vs. limpio'
)
plt.xticks(rotation=10)
plt.ylabel('N° palabras promedio')
plt.tight_layout()
plt.show()

---
## 7. Términos TF-IDF más relevantes por clase

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

if not VECTORIZER_PATH.exists():
    print('Ejecutando pipeline NLP para generar vectorizer...')
    run_nlp_pipeline()

vectorizer: TfidfVectorizer = joblib.load(VECTORIZER_PATH)
vocab = np.array(vectorizer.get_feature_names_out())

print(f'Vocabulario TF-IDF: {len(vocab)} términos')
print(f'N-gramas: {vectorizer.ngram_range}')
print(f'Max features: {vectorizer.max_features}')

In [ ]:
# Score TF-IDF promedio por clase
from scipy.sparse import issparse

label_map_inv = {0: LABEL_PROCRASTINATION, 1: LABEL_PRODUCTIVE}

# Reconstruir matriz TF-IDF por clase
tfidf_scores = {}
for label_int, label_str in label_map_inv.items():
    textos_clase = df_clean[df_clean['etiqueta'] == label_int]['texto_limpio'].astype(str)
    matrix = vectorizer.transform(textos_clase)
    mean_scores = np.asarray(matrix.mean(axis=0)).flatten()
    tfidf_scores[label_str] = mean_scores

TOP_TFIDF = 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label_str, color) in zip(axes, PALETTE.items()):
    scores = tfidf_scores[label_str]
    top_idx = np.argsort(scores)[::-1][:TOP_TFIDF]
    top_terms = vocab[top_idx]
    top_scores = scores[top_idx]

    bars = ax.barh(list(reversed(top_terms)), list(reversed(top_scores)),
                   color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'Top {TOP_TFIDF} términos TF-IDF — {label_str}', fontweight='bold')
    ax.set_xlabel('TF-IDF promedio')
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.suptitle('Términos más discriminativos por clase (TF-IDF)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Resultados del modelo (métricas CV y hold-out)

In [ ]:
if not METRICS_PATH.exists():
    print('No hay métricas aún. Ejecuta primero: python -m src.training.train_model')
else:
    with open(METRICS_PATH) as f:
        cv_metrics = json.load(f)
    with open(HOLDOUT_METRICS_PATH) as f:
        holdout_metrics = json.load(f)

    METRIC_KEYS = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC']

    comparativa = pd.DataFrame({
        'CV (train)':   {k: cv_metrics.get(k) for k in METRIC_KEYS},
        'Hold-out (test)': {k: holdout_metrics.get(k) for k in METRIC_KEYS},
    }).dropna(how='all').round(4)

    print(f"Modelo: {cv_metrics.get('model_name', 'N/A')}")
    print(f"Trainer: {cv_metrics.get('trainer', 'N/A')}")
    print(f"Train samples: {cv_metrics.get('n_samples_train', 'N/A')}")
    print(f"Test  samples: {holdout_metrics.get('n_samples_test', 'N/A')}")
    print()
    print(comparativa.to_string())

In [ ]:
if METRICS_PATH.exists():
    x = np.arange(len(comparativa.index))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, comparativa['CV (train)'],   width, label='CV (train)',      color='#3498db', alpha=0.85)
    bars2 = ax.bar(x + width/2, comparativa['Hold-out (test)'], width, label='Hold-out (test)', color='#e67e22', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(comparativa.index)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title(f'Métricas CV vs. Hold-out — {cv_metrics.get("model_name", "")}', fontweight='bold')
    ax.legend()
    ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, label='Umbral 0.8')

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                        ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

---
## 9. Matriz de confusión

In [ ]:
if not PER_CLASS_METRICS_PATH.exists():
    print('No hay métricas por clase. Ejecuta primero: python -m src.training.train_model')
else:
    with open(PER_CLASS_METRICS_PATH) as f:
        per_class = json.load(f)

    cm_data   = per_class['confusion_matrix']
    cm_matrix = np.array(cm_data['matrix'])
    cm_labels = cm_data['labels']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Matriz absoluta
    sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=cm_labels, yticklabels=cm_labels,
                linewidths=0.5, ax=axes[0], cbar=False)
    axes[0].set_title('Matriz de confusión (conteos)', fontweight='bold')
    axes[0].set_ylabel('Real')
    axes[0].set_xlabel('Predicho')

    # Matriz normalizada
    cm_norm = cm_matrix.astype(float) / cm_matrix.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=cm_labels, yticklabels=cm_labels,
                linewidths=0.5, ax=axes[1], cbar=False,
                vmin=0, vmax=1)
    axes[1].set_title('Matriz de confusión (normalizada)', fontweight='bold')
    axes[1].set_ylabel('Real')
    axes[1].set_xlabel('Predicho')

    plt.suptitle('Evaluación sobre hold-out (test set)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if PER_CLASS_METRICS_PATH.exists():
    report = per_class['classification_report']

    # Tabla por clase
    rows = []
    for label in cm_labels:
        if label in report:
            r = report[label]
            rows.append({
                'Clase': label,
                'Precision': round(r['precision'], 4),
                'Recall':    round(r['recall'], 4),
                'F1-score':  round(r['f1-score'], 4),
                'Support':   int(r['support']),
            })
    # Agregar promedios
    for avg_key in ['macro avg', 'weighted avg']:
        if avg_key in report:
            r = report[avg_key]
            rows.append({
                'Clase': avg_key,
                'Precision': round(r['precision'], 4),
                'Recall':    round(r['recall'], 4),
                'F1-score':  round(r['f1-score'], 4),
                'Support':   int(r.get('support', 0)),
            })

    df_report = pd.DataFrame(rows).set_index('Clase')
    print('Classification Report (hold-out):')
    print(df_report.to_string())

    # Gráfico de barras por clase
    metricas_clase = df_report.loc[cm_labels, ['Precision', 'Recall', 'F1-score']]
    metricas_clase.plot(kind='bar', figsize=(8, 4), color=['#9b59b6', '#1abc9c', '#f39c12'],
                        edgecolor='white', alpha=0.9)
    plt.title('Precision / Recall / F1 por clase', fontweight='bold')
    plt.ylabel('Score')
    plt.ylim(0, 1.15)
    plt.xticks(rotation=10)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

---
## 10. Distribución de probabilidades (efecto calibración)

Visualizamos la distribución de la probabilidad predicha para la clase *Productivo*.
Un modelo bien calibrado distribuye las probabilidades lejos del umbral 0.5 (alta confianza).
La calibración Platt/Sigmoid corrige los casos borderline (~0.53) que tenía el modelo base.

In [ ]:
from src.training.predict import predict_texts

textos_demo = list(df_raw['texto'])
etiquetas_reales = list(df_raw['etiqueta'])

resultados = predict_texts(textos_demo)
proba_prod = [r['probability'] if r['label'] == LABEL_PRODUCTIVE else 1 - r['probability']
              for r in resultados]
predicciones = [r['label'] for r in resultados]

df_proba = pd.DataFrame({
    'texto':          textos_demo,
    'etiqueta_real':  etiquetas_reales,
    'prediccion':     predicciones,
    'prob_productivo': proba_prod,
})
df_proba['correcto'] = df_proba['etiqueta_real'] == df_proba['prediccion']

print(f'Accuracy sobre dataset completo: {df_proba["correcto"].mean():.4f}')
print(f'Prob. promedio (Productivo)  : {df_proba[df_proba["etiqueta_real"]==LABEL_PRODUCTIVE]["prob_productivo"].mean():.4f}')
print(f'Prob. promedio (Procrastin.) : {df_proba[df_proba["etiqueta_real"]==LABEL_PROCRASTINATION]["prob_productivo"].mean():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma de probabilidades por clase real
for clase, color in PALETTE.items():
    subset = df_proba[df_proba['etiqueta_real'] == clase]['prob_productivo']
    axes[0].hist(subset, bins=12, alpha=0.65, color=color, label=clase, edgecolor='white')

axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Umbral 0.5')
axes[0].set_title('Distribución de P(Productivo) por clase real', fontweight='bold')
axes[0].set_xlabel('P(Productivo)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Scatter: probabilidad vs. corrección
colores_scatter = df_proba['correcto'].map({True: '#2ecc71', False: '#e74c3c'})
axes[1].scatter(
    range(len(df_proba)), df_proba['prob_productivo'],
    c=colores_scatter, alpha=0.75, s=60, edgecolors='white', linewidths=0.5
)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1, label='Umbral 0.5')
axes[1].set_title('Confianza por muestra (verde=correcto, rojo=error)', fontweight='bold')
axes[1].set_xlabel('Índice de muestra')
axes[1].set_ylabel('P(Productivo)')
axes[1].legend()

plt.suptitle('Análisis de probabilidades del modelo calibrado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Casos borderline (probabilidad cercana a 0.5)
borderline = df_proba[df_proba['prob_productivo'].between(0.4, 0.6)].sort_values('prob_productivo')

print(f'Casos borderline (0.4 < P < 0.6): {len(borderline)}')
if not borderline.empty:
    print()
    for _, row in borderline.iterrows():
        status = 'OK' if row['correcto'] else 'ERROR'
        print(f'[{status}] Real: {row["etiqueta_real"]:15s} | Pred: {row["prediccion"]:15s} | P={row["prob_productivo"]:.3f}')
        print(f'  "{row["texto"][:80]}..."')
        print()

---
## Resumen

| Aspecto | Hallazgo |
|---------|----------|
| **Dataset** | 60 entradas balanceadas (30 Productivo / 30 Procrastinación) |
| **Longitud** | ~20 palabras promedio por entrada; ambas clases similares |
| **Vocabulario NLP** | TF-IDF con hasta 1000 términos, n-gramas (1,2) |
| **Discriminación** | Términos como *terminé*, *avancé*, *completé* vs. *aplacé*, *distraje*, *pospuse* |
| **Modelo** | Ensemble (XGBoost/RF/LightGBM/GBC) seleccionado por F1-weighted |
| **Calibración** | Platt scaling (`sigmoid`) aplicado post-tuning |

> **Limitación principal:** con solo 60 muestras, los resultados del hold-out pueden variar según el split. Ampliar el dataset es la mejora de mayor impacto para este proyecto.